# Deploying Flan-T5-XXL in SageMaker

Setting up the role and S3 bucket that we will need later

In [2]:
import sagemaker
import boto3

sess = sagemaker.Session()
sagemaker_session_bucket = sess.default_bucket()
role = sagemaker.get_execution_role() 

[Optional] Deleting the `model.tar.gz` file, if one exists

In [5]:
import os
filePath = 'model/model.tar.gz'

if os.path.exists(filePath):
    os.remove(filePath)

In [6]:
model_name = "flan-t5-xxl"

Creating a ne w`model.tar.gz` file and uploading it to S3

In [7]:
%cd model
!tar zcvf model.tar.gz *
s3_location = f"s3://{sess.default_bucket()}/{model_name}/model.tar.gz"
!aws s3 cp model.tar.gz $s3_location
%cd ..

/root/llm-playground/model
code/
code/requirements.txt
code/.ipynb_checkpoints/
code/.ipynb_checkpoints/requirements-checkpoint.txt
code/inference.py
upload: ./model.tar.gz to s3://sagemaker-us-east-2-811196786605/flan-t5-xxl/model.tar.gz
/root/llm-playground


Creating the Hugging Face Model, indicating the package versions we want to use and the S£ location with the inference code

In [8]:
from sagemaker.huggingface.model import HuggingFaceModel

huggingface_model = HuggingFaceModel(
    model_data=s3_location,
    role=role,
    transformers_version="4.17",
    pytorch_version="1.10",
    py_version='py38',
)

Deploying the model to an endpoint

In [9]:
from sagemaker.utils import name_from_base

endpoint_name = name_from_base(model_name)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.4xlarge",
    endpoint_name=endpoint_name,
)

-----------!

!!!NOTE: Even after the endpoint has been deployed, we still need to wait 1-2 minutes before we can start using it. That's because the model is downloading from the HF Model Hub and due to its size it won't be quite finished when the endpoint is deployed.

In [11]:
predictor.endpoint_name

'flan-t5-xxl-2023-05-02-17-35-05-186'

In [12]:
prompt = """Answer the following question by reasoning step by step.
The cafeteria had 23 apples. If they used 20 for lunch, and bought 6 more, how many apples do they have now?"""                                              

In [15]:
data = {
    "inputs": prompt,
    "min_length": 20,
    "max_length": 50,
    "do_sample": True,
    "temperature": 0.6,
}

res = predictor.predict(data=data)
print(res)

They had 23 - 20 = 3 apples after the lunch. After buying 6, they have 3 + 6 = 9 apples. Therefore, the answer is 9.


# Langchain Sagemaker Integration

In [1]:
!pip install langchain

  Using cached langchain-0.0.155-py3-none-any.whl (727 kB)
  Using cached aiohttp-3.8.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.0 MB)
  Using cached async_timeout-4.0.2-py3-none-any.whl (5.8 kB)
  Using cached SQLAlchemy-2.0.12-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.7 MB)
  Using cached numexpr-2.8.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (380 kB)
  Using cached openapi_schema_pydantic-1.2.4-py3-none-any.whl (90 kB)
  Using cached dataclasses_json-0.5.7-py3-none-any.whl (25 kB)
  Using cached frozenlist-1.3.3-cp39-cp39-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (158 kB)
  Using cached yarl-1.9.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (269 kB)
  Using cached aiosignal-1.3.1-py3-none-any.whl (7.6 kB)
  Using cached multidict-6.0.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (114 kB)
  Using cached marshmallow-3.19.0-py3-none-any.whl (49 kB)
  Using cached t

In [4]:
from langchain.docstore.document import Document

In [5]:
example_doc_1 = """
Peter and Elizabeth took a taxi to attend the night party in the city. While in the party, Elizabeth collapsed and was rushed to the hospital.
Since she was diagnosed with a brain injury, the doctor told Peter to stay besides her until she gets well.
Therefore, Peter stayed with her at the hospital for 3 days without leaving.
"""

docs = [
    Document(
        page_content=example_doc_1,
    )
]

In [6]:
from typing import Dict

from langchain import PromptTemplate, SagemakerEndpoint
from langchain.llms.sagemaker_endpoint import ContentHandlerBase
from langchain.chains.question_answering import load_qa_chain
import json
import boto3

In [7]:
endpoint_name='flan-t5-xxl-2023-05-02-17-35-05-186'

In [8]:
query = """How long was Elizabeth hospitalized?
"""

prompt_template = """Use the following pieces of context to answer the question at the end.

{context}

Question: {question}
Answer:"""

In [9]:
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [10]:
class ContentHandler(ContentHandlerBase):
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs: Dict) -> bytes:
        input_str = json.dumps({prompt: prompt, **model_kwargs})
        return input_str.encode('utf-8')
    
    def transform_output(self, output: bytes) -> str:
        response_json = json.loads(output.read().decode("utf-8"))
        return response_json[0]["generated_text"]


In [11]:
content_handler = ContentHandler()

In [12]:
for profile in boto3.session.Session().available_profiles:
    print(profile)

default
dev


In [13]:
chain = load_qa_chain(
    llm=SagemakerEndpoint(
        endpoint_name=endpoint_name, 
        credentials_profile_name="dev", 
        region_name="us-east-2", 
        model_kwargs={"temperature":1e-10},
        content_handler=content_handler
    ),
    prompt=PROMPT
)

ValidationError: 1 validation error for SagemakerEndpoint
content_handler
  instance of LLMContentHandler expected (type=type_error.arbitrary_type; expected_arbitrary_type=LLMContentHandler)